In [ ]:
import os
import math
import json
import joblib
from datetime import datetime

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
import lightgbm as lgb
import optuna

# -------------------------
# Configuration / Constants
# -------------------------
SEED = 42
np.random.seed(SEED)

# Paths (change if needed)
DEFAULT_CSV_PATHS = [
    "/mnt/data/gbm_project_dataset (1).csv",  # uploaded file path (developer note)
    "gbm_dataset.csv",                        # local generated dataset file
    "/mnt/data/gbm_dataset.csv"
]
OUTPUT_DIR = "gbm_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Business penalty: factor for underprediction relative to overprediction
UNDERPRED_PENALTY = 3.0

# Optuna tuning configuration
N_TRIALS = 50
N_FOLDS = 1  # simple train/valid split; for production, use CV or time-series CV as needed
EARLY_STOPPING_ROUNDS = 50
NROUNDS = 5000  # upper bound for boosting rounds
VERBOSE = -1

# --------------
# Helper methods
# --------------

def find_dataset():
    """Return first existing dataset path from DEFAULT_CSV_PATHS or raise FileNotFoundError."""
    for p in DEFAULT_CSV_PATHS:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(
        f"No dataset found. Please place CSV at one of: {DEFAULT_CSV_PATHS}"
    )

def load_data(path=None, target_col="target", test_size=0.2):
    """Load CSV and split into train/validation sets."""
    if path is None:
        path = find_dataset()
    print(f"Loading dataset from: {path}")
    df = pd.read_csv(path)
    if target_col not in df.columns:
        raise ValueError(f"Target column '{target_col}' not found in dataset.")
    X = df.drop(columns=[target_col])
    y = df[target_col].values
    # For simplicity we will one-hot encode categorical columns that appear integer-like (small cardinality).
    # If dataset already encoded, skip.
    # Detect categorical columns by dtype or name pattern
    cat_cols = [c for c in X.columns if c.startswith("cat_") or pd.api.types.is_integer_dtype(X[c])]
    # But only treat as categorical if unique values <= 50
    cat_cols = [c for c in cat_cols if X[c].nunique() <= 50]
    if len(cat_cols) > 0:
        X = pd.get_dummies(X, columns=cat_cols, drop_first=True)
    # Split
    X_train, X_valid, y_train, y_valid = train_test_split(
        X, y, test_size=test_size, random_state=SEED
    )
    return X_train, X_valid, y_train, y_valid

# -----------------------------------
# Custom asymmetric squared loss
# -----------------------------------
# Business requirement: under-forecasting (prediction < actual) is 3x more costly than over-forecasting.
# Define loss per sample:
#   r = y_true - y_pred
#   w = UNDERPRED_PENALTY if r > 0 else 1
#   L = w * r^2
#
# Grad and Hess:
#   dL/dpred = -2 * w * r   -> gradient (wrt prediction)
#   d2L/dpred2 = 2 * w       -> hessian (constant per-sample)
#
# Note: w depends on sign(r). It's piecewise constant; gradient/hessian computed per-sample is acceptable for tree-based boosters.

def asymmetric_grad_hess(preds: np.ndarray, dtrain: lgb.Dataset):
    """
    LightGBM custom objective (returns grad, hess)
    preds : raw predictions (not necessarily transformed); LightGBM uses raw values for regression
    dtrain: lgb.Dataset (provides labels)
    """
    y = dtrain.get_label()
    # residual r = y - preds
    r = y - preds
    # weight per sample: UNDERPRED_PENALTY if underprediction (r > 0), else 1
    w = np.where(r > 0, UNDERPRED_PENALTY, 1.0)
    grad = -2.0 * w * r  # derivative of L wrt preds
    hess = 2.0 * w       # second derivative
    return grad, hess

def asymmetric_eval_metric(preds: np.ndarray, dtrain: lgb.Dataset):
    """
    Custom eval function for LightGBM (returns name, value, is_higher_better)
    This returns the mean asymmetric loss (not root or scaled).
    """
    y = dtrain.get_label()
    r = y - preds
    w = np.where(r > 0, UNDERPRED_PENALTY, 1.0)
    loss = (w * (r ** 2)).mean()
    return "asym_mse", loss, False

# -------------------------
# Baseline model (MSE)
# -------------------------
def train_baseline(X_train, X_valid, y_train, y_valid, seed=SEED):
    """Train a default LightGBM regressor (baseline) using MSE loss and default hyperparams."""
    dtrain = lgb.Dataset(X_train, label=y_train)
    dvalid = lgb.Dataset(X_valid, label=y_valid, reference=dtrain)
    params = {
        "objective": "regression",
        "metric": "rmse",
        "verbosity": -1,
        "seed": seed,
    }
    print("Training baseline LightGBM (default params)...")
    model = lgb.train(
        params,
        dtrain,
        num_boost_round=1000,
        valid_sets=[dtrain, dvalid],
        valid_names=["train", "valid"],
        callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)]
    )
    return model

# -------------------------
# Optuna objective
# -------------------------
def optuna_objective(trial, X_train, X_valid, y_train, y_valid):
    """
    Optuna objective: suggests LightGBM hyperparams, trains using custom objective,
    and returns validation asymmetric loss (lower is better).
    """
    param = {
        "objective": asymmetric_grad_hess,  # Pass custom objective function here
        "verbosity": -1,
        "metric": "None",  # we'll use custom eval metric
        "boosting_type": "gbdt",
        "seed": SEED,
        "learning_rate": trial.suggest_loguniform("learning_rate", 1e-3, 0.2),
        "num_leaves": trial.suggest_int("num_leaves", 16, 512),
        "max_depth": trial.suggest_int("max_depth", 3, 16),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 5, 200),
        "feature_fraction": trial.suggest_uniform("feature_fraction", 0.4, 1.0),
        "bagging_fraction": trial.suggest_uniform("bagging_fraction", 0.4, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 0, 10),
        "lambda_l1": trial.suggest_loguniform("lambda_l1", 1e-8, 10.0),
        "lambda_l2": trial.suggest_loguniform("lambda_l2", 1e-8, 10.0),
    }

    dtrain = lgb.Dataset(X_train, label=y_train)
    dvalid = lgb.Dataset(X_valid, label=y_valid, reference=dtrain)

    # train with custom objective
    model = lgb.train(
        param,
        dtrain,
        num_boost_round=NROUNDS,
        valid_sets=[dvalid],
        valid_names=["valid"],
        feval=asymmetric_eval_metric,
        callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)]
    )

    # retrieve best asymmetric validation loss from evals_result
    # LightGBM records the custom eval under 'valid' -> 'asym_mse'
    # The evals_result attribute is part of the returned model object.
    try:
        best_iter = model.best_iteration
        asym_vals = model.evals_result["valid"]["asym_mse"]
        best_val = asym_vals[best_iter - 1] if best_iter - 1 < len(asym_vals) else asym_vals[-1]
    except Exception:
        # compute custom loss on validation preds as fallback
        preds = model.predict(X_valid, num_iteration=model.best_iteration)
        y = y_valid
        r = y - preds
        w = np.where(r > 0, UNDERPRED_PENALTY, 1.0)
        best_val = (w * r ** 2).mean()

    # optuna minimizes by default
    trial.set_user_attr("best_iteration", model.best_iteration)
    return float(best_val)

# -------------------------
# Training + Evaluation flow
# -------------------------
def run_full_pipeline():
    # Load data
    X_train, X_valid, y_train, y_valid = load_data()
    print(f"Train shape: {X_train.shape}, Valid shape: {X_valid.shape}")

    # Baseline model
    baseline_model = train_baseline(X_train, X_valid, y_train, y_valid)
    # baseline predictions
    preds_baseline = baseline_model.predict(X_valid, num_iteration=baseline_model.best_iteration)
    baseline_mae = mean_absolute_error(y_valid, preds_baseline)
    baseline_rmse = math.sqrt(mean_squared_error(y_valid, preds_baseline))
    # baseline custom loss
    r_base = y_valid - preds_baseline
    w_base = np.where(r_base > 0, UNDERPRED_PENALTY, 1.0)
    baseline_asym = (w_base * (r_base ** 2)).mean()

    print("Baseline metrics:")
    print(f"  MAE: {baseline_mae:.6f}")
    print(f"  RMSE: {baseline_rmse:.6f}")
    print(f"  Asymmetric MSE (custom): {baseline_asym:.6f}")

    # Optuna tuning for custom-loss model
    print(f"\nStarting Optuna tuning ({N_TRIALS} trials) to minimize asymmetric loss...")
    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    func = lambda trial: optuna_objective(trial, X_train, X_valid, y_train, y_valid)
    study.optimize(func, n_trials=N_TRIALS, show_progress_bar=True)

    print("Best trial:")
    print(f"  Value (best asymmetric loss): {study.best_value:.6f}")
    print(f"  Params: {study.best_params}")

    # Train final model using best params and custom objective
    best_params = study.best_params
    # Ensure default keys
    best_params.update({
        "objective": asymmetric_grad_hess, # Set custom objective for final model
        "verbosity": -1,
        "metric": "None",
        "boosting_type": best_params.get("boosting_type", "gbdt"),
        "seed": SEED
    })
    dtrain = lgb.Dataset(X_train, label=y_train)
    dvalid = lgb.Dataset(X_valid, label=y_valid, reference=dtrain)
    final_model = lgb.train(
        best_params,
        dtrain,
        num_boost_round=NROUNDS,
        valid_sets=[dvalid],
        feval=asymmetric_eval_metric,
        callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)] # Fixed
    )

    # Final predictions and metrics
    preds_final = final_model.predict(X_valid, num_iteration=final_model.best_iteration)
    final_mae = mean_absolute_error(y_valid, preds_final)
    final_rmse = math.sqrt(mean_squared_error(y_valid, preds_final))
    r_final = y_valid - preds_final
    w_final = np.where(r_final > 0, UNDERPRED_PENALTY, 1.0)
    final_asym = (w_final * (r_final ** 2)).mean()

    print("\nTuned custom-loss model metrics:")
    print(f"  MAE: {final_mae:.6f}")
    print(f"  RMSE: {final_rmse:.6f}")
    print(f"  Asymmetric MSE (custom): {final_asym:.6f}")

    # Save models and results
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    baseline_path = os.path.join(OUTPUT_DIR, f"baseline_lgb_{timestamp}.pkl")
    final_path = os.path.join(OUTPUT_DIR, f"custom_lgb_tuned_{timestamp}.pkl")
    joblib.dump(baseline_model, baseline_path)
    joblib.dump(final_model, final_path)

    # Save predictions for analysis
    out_df = pd.DataFrame({
        "y_true": y_valid,
        "pred_baseline": preds_baseline,
        "pred_custom_tuned": preds_final,
        "resid_baseline": y_valid - preds_baseline,
        "resid_custom": y_valid - preds_final
    })
    preds_csv = os.path.join(OUTPUT_DIR, f"predictions_compare_{timestamp}.csv")
    out_df.to_csv(preds_csv, index=False)

    # Save study best params and summary report
    summary = {
        "seed": SEED,
        "underprediction_penalty": UNDERPRED_PENALTY,
        "n_trials": N_TRIALS,
        "best_value_asym_loss": float(study.best_value),
        "best_params": study.best_params,
        "baseline_metrics": {
            "mae": float(baseline_mae),
            "rmse": float(baseline_rmse),
            "asym_mse": float(baseline_asym)
        },
        "final_metrics": {
            "mae": float(final_mae),
            "rmse": float(final_rmse),
            "asym_mse": float(final_asym)
        },
        "files": {
            "baseline_model": baseline_path,
            "tuned_model": final_path,
            "predictions_csv": preds_csv
        }
    }
    summary_path = os.path.join(OUTPUT_DIR, f"summary_{timestamp}.json")
    with open(summary_path, "w") as f:
        json.dump(summary, f, indent=2)

    # Print succinct report
    print("\nSaved outputs:")
    print(f"  Baseline model: {baseline_path}")
    print(f"  Tuned custom model: {final_path}")
    print(f"  Predictions CSV: {preds_csv}")
    print(f"  Summary JSON: {summary_path}")

    # Also produce a simple text summary file
    report_txt = os.path.join(OUTPUT_DIR, f"report_{timestamp}.txt")
    with open(report_txt, "w") as r:
        r.write("GBM Custom Loss Tuning Report\n")
        r.write("============================= \n\n")
        r.write(f"Dataset path: {find_dataset()}\n")
        r.write(f"Seed: {SEED}\n")
        r.write(f"Underprediction penalty: {UNDERPRED_PENALTY}\n\n")
        r.write("Baseline metrics:\n")
        r.write(f"  MAE: {baseline_mae:.6f}\n")
        r.write(f"  RMSE: {baseline_rmse:.6f}\n")
        r.write(f"  Asymmetric MSE: {baseline_asym:.6f}\n\n")
        r.write("Tuned custom model metrics:\n")
        r.write(f"  MAE: {final_mae:.6f}\n")
        r.write(f"  RMSE: {final_rmse:.6f}\n")
        r.write(f"  Asymmetric MSE: {final_asym:.6f}\n\n")
        r.write("Optuna best params:\n")
        r.write(json.dumps(study.best_params, indent=2))
    print(f"Report saved: {report_txt}")

    return summary

# --------------
# Entrypoint
# --------------
if __name__ == "__main__":
    summary = run_full_pipeline()
    print("\nDone. Summary (top-level):")
    print(json.dumps({
        "best_asym_loss": summary["best_value_asym_loss"],
        "baseline_asym_loss": summary["baseline_metrics"]["asym_mse"],
        "final_asym_loss": summary["final_metrics"]["asym_mse"],
        "best_params": summary["best_params"]
    }, indent=2))